# 🤖 FutbolIA — Pipeline Profesional de Análisis Táctico

**Modelo:** YOLO26L (Ultralytics) + ByteTrack  
**Repositorio:** [github.com/HectrorrVas/FutbolIA](https://github.com/HectrorrVas/FutbolIA)  
**Google Drive (modelo + videos):** [Abrir carpeta](https://drive.google.com/drive/folders/15nc9FGu59d2YMkjfg_UXtFtHlaZ9_4ci)

---

### ▶️ Instrucciones
Ejecuta las celdas **en orden** de arriba hacia abajo. El proceso es completamente automático.

## Celda 1 — Clonar el repositorio e instalar dependencias

In [ ]:
# 1. Clonar el repositorio desde GitHub
!rm -rf /content/FutbolIA
!git clone https://github.com/HectrorrVas/FutbolIA.git /content/FutbolIA

# 2. Instalar dependencias del proyecto
!pip install -q ultralytics imageio-ffmpeg tqdm opencv-python numpy

print("\n✅ Repositorio clonado y dependencias instaladas.")

## Celda 2 — Montar Google Drive y copiar modelo + video

Esta celda monta tu Google Drive y copia automáticamente los archivos a las carpetas correctas:
- `model/best.pt` → Pesos del modelo YOLO26L
- `clips/original/clip_01.mp4` → Video a procesar

> ⚠️ Asegúrate de que los archivos estén en tu Google Drive en la carpeta compartida.

In [ ]:
import shutil
import os
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  CONFIGURACIÓN: ajusta estas rutas si tus archivos están en otra subcarpeta
# ─────────────────────────────────────────────────────────────────────────────
DRIVE_FOLDER  = "/content/drive/MyDrive/FutbolIA"  # Carpeta raíz en tu Drive
MODEL_FILE    = "best.pt"                           # Nombre del archivo del modelo
VIDEO_FILE    = "clip_01.mp4"                       # Nombre del video a procesar
# ─────────────────────────────────────────────────────────────────────────────

# Crear carpetas destino en Colab
os.makedirs("/content/FutbolIA/model",           exist_ok=True)
os.makedirs("/content/FutbolIA/clips/original",  exist_ok=True)
os.makedirs("/content/FutbolIA/clips/processed", exist_ok=True)
os.makedirs("/content/FutbolIA/output/heatmaps", exist_ok=True)

# Copiar modelo
src_model = f"{DRIVE_FOLDER}/{MODEL_FILE}"
dst_model = f"/content/FutbolIA/model/{MODEL_FILE}"
if os.path.exists(src_model):
    shutil.copy2(src_model, dst_model)
    print(f"✅ Modelo copiado: {dst_model}")
else:
    print(f"❌ Modelo NO encontrado en: {src_model}")
    print(f"   Ajusta la variable DRIVE_FOLDER en esta celda.")

# Copiar video de entrada
src_video = f"{DRIVE_FOLDER}/{VIDEO_FILE}"
dst_video = f"/content/FutbolIA/clips/original/{VIDEO_FILE}"
if os.path.exists(src_video):
    shutil.copy2(src_video, dst_video)
    sz = os.path.getsize(dst_video) / (1024*1024)
    print(f"✅ Video copiado: {dst_video}  ({sz:.1f} MB)")
else:
    print(f"❌ Video NO encontrado en: {src_video}")
    print(f"   Ajusta la variable VIDEO_FILE o DRIVE_FOLDER en esta celda.")

## Celda 3 — Ejecutar el pipeline de análisis táctico

Esta celda procesará el video completo con la GPU de Colab y generará los 4 videos de salida:
- `clip_01_main.mp4` — Vista general + mapa 2D
- `clip_01_equipoA.mp4` — Análisis táctico Equipo A
- `clip_01_equipoB.mp4` — Análisis táctico Equipo B
- `clip_01_grid.mp4` — **Grid 2x2 (todas las vistas combinadas)**

In [ ]:
!PYTHONPATH=/content/FutbolIA python /content/FutbolIA/src/detect_video.py

## Celda 4 — Visualizar los videos en Colab

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

def show_video(path, title=""):
    if not os.path.exists(path):
        print(f"⚠️  No encontrado: {path}")
        return
    sz = os.path.getsize(path) / (1024*1024)
    data = open(path, 'rb').read()
    url  = "data:video/mp4;base64," + b64encode(data).decode()
    display(HTML(f"""
        <h3 style='color:#ddd;font-family:sans-serif'>{title} &nbsp;<small style='color:#888'>({sz:.1f} MB)</small></h3>
        <video width='900' controls style='border-radius:8px;margin-bottom:20px'>
            <source src='{url}' type='video/mp4'>
        </video>
    """))

base = "/content/FutbolIA/clips/processed"

show_video(f"{base}/clip_01_grid.mp4",    "⚽ Grid 2x2 — Todas las vistas")
show_video(f"{base}/clip_01_main.mp4",    "📊 Vista General + Mapa 2D")
show_video(f"{base}/clip_01_equipoA.mp4", "🔵 Análisis Táctico — Equipo A")
show_video(f"{base}/clip_01_equipoB.mp4", "🔴 Análisis Táctico — Equipo B")

## Celda 5 — Descargar los videos procesados a tu PC

In [ ]:
import os
from google.colab import files

base = "/content/FutbolIA/clips/processed"
videos = [
    (f"{base}/clip_01_grid.mp4",    "Grid 2x2"),
    (f"{base}/clip_01_main.mp4",    "Main"),
    (f"{base}/clip_01_equipoA.mp4", "Equipo A"),
    (f"{base}/clip_01_equipoB.mp4", "Equipo B"),
]

for path, label in videos:
    if os.path.exists(path):
        print(f"⬇️  Descargando: {label}...")
        files.download(path)
    else:
        print(f"⚠️  No encontrado: {label}")